# Session 23: Ship it: reproducible work, a small app and responsible use

Seed a draw, measure fairness with pandas, save a model, write an app file, and complete a model card with a data protection checklist for the Nile Microfinance loans. Apply the same steps to your own capstone model.

**Student version.** Run the cells from the top. Each cell that starts with '# Your turn' is yours to complete. Run it, then run the check cell after it: the check tells you if you are right and what to fix.

In [ ]:
# Run this cell first. It puts the course data files next to your notebook.
import os, zipfile, urllib.request
DATA_ZIP_URL = ""   # your tutor may give you a link; leave empty to upload data.zip instead
if not os.path.exists("orders.csv"):
    if DATA_ZIP_URL:
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    else:
        from google.colab import files
        print("Choose the file data.zip from your computer")
        uploaded = files.upload()
        os.replace(list(uploaded)[0], "data.zip")   # works even if the browser renamed it to data (1).zip
    zipfile.ZipFile("data.zip").extractall(".")
print("Data ready:", sorted(f for f in os.listdir(".") if f.endswith((".csv", ".db"))))

## Task 1. Same seed, same sample
Load `loans.csv`. Draw five loan ids with `loans.sample(5, random_state=42)` twice, keep the lists in `s1` and `s2`, and print the library versions.

In [ ]:
import numpy as np
import pandas as pd

loans = pd.read_csv('loans.csv')
# Your turn: draw 5 loan ids twice with random_state=42, into s1 and s2
# Your turn: print numpy and pandas versions


In [ ]:
_missing = [v for v in ('s1', 's2',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert len(s1) == 5 and s1 == s2, 's1 and s2 should each hold 5 loan ids and be equal. Use random_state=42 both times and take .loan_id.tolist()'
print('Task 1 correct')

## Task 2. Default rate by group
Tidy `district` with `.str.strip().str.title()`. Then store the default rate by gender in `by_gender` and by district in `by_district`.

In [ ]:
# Your turn: tidy the district column
# Your turn: store the mean of default by gender in by_gender and by district in by_district


In [ ]:
_missing = [v for v in ('by_district', 'by_gender',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert abs(by_gender['F'] - 0.121) < 0.003, 'by_gender["F"] should be about 0.121. Take the mean of default for each gender, after tidying district'
assert by_district.idxmax() == 'Masaka', 'Masaka should have the highest default rate in by_district. Did you tidy district with strip and title first?'
print('Task 2 correct')

## Task 3. A rule and its flag rate
Drop loans with no income into `known`. Flag a loan when the monthly instalment (`loan_amount_ugx / term_months`) is over 35 percent of income. Store the flag rate by gender in `flag_by_gender`.

In [ ]:
# Your turn: known = the loans that have a monthly income
# Your turn: work out each loan's monthly instalment as a share of income
# Your turn: flag = 1 when that share is over 0.35, otherwise 0
# Your turn: store the mean flag by gender in flag_by_gender


In [ ]:
_missing = [v for v in ('flag_by_gender',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert 0.30 <= flag_by_gender['F'] <= 0.40, 'The female flag rate should be between 0.30 and 0.40. Divide the instalment (loan_amount_ugx / term_months) by income and flag above 0.35'
assert abs(flag_by_gender['F'] - flag_by_gender['M']) < 0.03, 'The male and female flag rates should be within 0.03 of each other. Check you used the same rule for both'
print('Task 3 correct')

## Task 4. Look for a proxy
Give every loan its occupation's default rate with `transform('mean')`. Flag loans above 0.125 in `flag2`. Store the flag rate by gender in `flag2_by_gender`.

In [ ]:
# Your turn: give every loan the default rate of its occupation, using groupby and transform
# Your turn: flag2 = 1 when that rate is above 0.125, otherwise 0, stored as a column of known
# Your turn: store the mean of flag2 by gender in flag2_by_gender


In [ ]:
_missing = [v for v in ('flag2_by_gender',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
gap = flag2_by_gender['M'] - flag2_by_gender['F']
assert 0.10 < gap < 0.25, 'The gap (male minus female flag2 rate) should be between 0.10 and 0.25. Use the occupation default rate with transform and flag above 0.125'
print('Task 4 correct')

## Task 5. Save and load a model
Fit a pipeline on four features, save it with joblib, load it back into `loaded`, and store the default probability of the first loan in `p`. Print it.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import joblib

features = ['monthly_income_ugx', 'loan_amount_ugx', 'term_months', 'previous_late_payments']
# Your turn: build model, a pipeline of an imputer, a scaler and a logistic regression, and fit it
# Your turn: save it to the file loan_model.joblib, then load that file into loaded
# Your turn: store the default probability of the first loan in p, and print it


In [ ]:
_missing = [v for v in ('p',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
import os
assert os.path.exists('loan_model.joblib'), 'loan_model.joblib is missing. Save the model with joblib.dump'
import joblib
assert 0.0 <= p <= 1.0, 'p should be a probability between 0 and 1. Use predict_proba and take the column for default'
assert 0.01 < p < 0.6, 'p should be between 0.01 and 0.6 for the first loan. Fit on the four features and predict from the loaded model'
print('Task 5 correct')

## Task 6. Write the app file
Copy the app text from the Session 23 slide into a string called `app_code` and write it to a file named `app.py`. Then read it line by line and find the line that loads the model and the line that shows the probability. To deploy it, put `app.py`, `loan_model.joblib` and `requirements.txt` in a public GitHub repository. Never include data with personal details.

In [ ]:
# Your turn: copy the app text from the slide into a string called app_code
# Your turn: write app_code to a file named app.py


In [ ]:
import os
assert os.path.exists('app.py'), 'app.py is missing. Write app_code to a file named app.py'
text = open('app.py').read()
assert 'loan_model.joblib' in text, 'app.py should load loan_model.joblib'
assert 'predict_proba' in text, 'app.py should call predict_proba to show the probability'
print('Task 6 correct')

## Task 7. Complete the model card
Double click this cell and replace each dots line with your own words.

**Purpose:** ...

**Data:** ... (source, years, what you removed and why)

**Metrics:** ... (for this loan model the AUC is about 0.76; quote that, or your own capstone figure)

**Limits:** ... (include the gender flag-rate gap you measured in Task 4)

**Who must review:** ... (a role, and a date)

**Data protection checklist:** one line for each of the seven principles in the Session 23 table, saying what you do or still need to do. Mark any line you cannot answer yet.

## Task 8. Reflect
Open `loans.csv` in Google Sheets. Build a PivotTable of the average of `default` by `gender` and check it against Task 2. Then write one sentence: what does the gap in Task 4 show, and what does it leave open?